# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My Rule is:**
**A page is worth reviewing if it actually shows up in the search (enough impressions, ranked in the top 20), and its click-through rate is below the median CTR of other pages in its OWN position tier (not one flat number for everyone)**

In [4]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()
print(f"visible pages (n): {len(visible):,}")

visible pages (n): 12,023


In [5]:
tier_order = ['top_3', 'page_1', 'striking']
signal_a = visible[visible['position_tier'].isin(tier_order)].groupby('position_tier')['ctr'].agg(n='count', median_ctr='median', mean_ctr='mean')
signal_a = signal_a.reindex(tier_order)
print(signal_a)

                  n  median_ctr  mean_ctr
position_tier                            
top_3           458        0.20  0.346572
page_1         7064        0.24  0.338808
striking       4485        0.17  0.266798


**What I did:** Signal A (CTR by position tier)

I checked that whether the pages that actually rank high  get more clicks or not. I did this because FlyRank's real **needs_ctr_fix** flag assumes CTR should only be judged compared to other pages at the same rank, not by one number for everyone.

I grouped my visible pages (12,023) into three tiers:

 top_3 (ranks 1-3),

 page_1 (ranks 4-10), 
 
 striking (ranks 11-20), and looked at the typical CTR in each group.

My results were :

 top_3: n=458, median CTR = 0.20

 page_1: n=7,064, median CTR = 0.24
 
 striking: n=4,485, median CTR = 0.17

**Striking** is clearly the worst of the three ranks , which is what I expected (lower rank, 
fewer clicks). But **top_3** actually came out lower than **page_1**, which is backwards 
from what I expected. **top_3** only has 458 pages as compared to **page_1's** 7,064 pages, so the
result are on a more tenuous position.

**Verdict:** (**MIXED**) Position does matter (striking is clearly worse), but it is not 
a clean straight line where every step up in rank means better CTR. Because of this, 
I'll compare each page against the median CTR of its OWN tier, instead of one flat 
number for the whole dataset.

In [6]:
tier_median_ctr = visible.groupby('position_tier')['ctr'].transform('median')
visible['tier_median_ctr'] = tier_median_ctr
visible['ctr_gap'] = tier_median_ctr - visible['ctr']

impression_order = ['low', 'moderate', 'good', 'excellent']
signal_b = visible.groupby('impression_tier')['ctr_gap'].agg(n='count', mean_gap='mean', median_gap='median')
signal_b = signal_b.reindex(impression_order)
print(signal_b)

                      n  mean_gap  median_gap
impression_tier                              
low                 NaN       NaN         NaN
moderate         5717.0 -0.062161        0.03
good             5463.0 -0.134953       -0.03
excellent         843.0 -0.127657       -0.03


**What I did:**
 Signal B (does more search volume means more room for CTR improvement?)

I checked whether pages with more impressions actually have a bigger CTR gap (more room to improve), the way FlyRank's is_quick_win logic assumes, that bigger-audience pages are the best bet because fixing them can reach more people.

I used ctr_gap = tier_median_ctr - page's own ctr, so a positive gap means the page is underperforming its tier, and a negative gap means it's already beating its tier's median.

My results were:

 low = no data (filtered out by my visibility rule, since low-impression pages don't reach 500 impressions), 

 moderate: n=5,717, mean_gap = -0.06,
 
 good: n=5,463, mean_gap = -0.13, 

 excellent: n=843, mean_gap = -0.13

As impressions go up, the mean gap gets more negative, not more positive. That means pages with more traffic tend to already be doing better than their tier's median, not worse. That's the opposite of what "more volume = more room to improve" assumes.

**Verdict:** (**OPPOSITE**) High-volume pages tend to already outperform their peers, not 
sit on hidden opportunity. Because of this, I won't multiply my score by impressions that would push already-good pages higher instead of surfacing real underperformers. I'll only use impressions_90d >= 500 as a gate (is this page worth a reviewer's time at all), not as a score booster.

**Reason code:** ctr_below_tier_peers (the page's CTR is below the median CTR of other pages in its own position tier).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
visible['score'] = visible['ctr_gap'].clip(lower=0)
visible['reason_code'] = 'ctr_below_tier_peers'
visible['action'] = 'review_title_and_meta_description'

ranked = visible.sort_values(['score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)

print(f"eligible pages: {len(ranked):,}")
print(f"pages that scored above 0: {(ranked['score'] > 0).sum():,}")

eligible pages: 12,023
pages that scored above 0: 5,888


In [8]:
import os

out_cols = ['content_id', 'client_id', 'position_tier', 'avg_position', 'ctr',
            'tier_median_ctr', 'ctr_gap', 'score', 'reason_code', 'action',
            'impressions_90d', 'search_volume']

os.makedirs('../outputs', exist_ok=True)
ranked[out_cols].to_csv('../outputs/baseline_action_score.csv', index=False)
print("wrote work/outputs/baseline_action_score.csv")

ranked[out_cols].head(10)

wrote work/outputs/baseline_action_score.csv


,content_id,client_id,position_tier,avg_position,ctr,tier_median_ctr,ctr_gap,score,reason_code,action,impressions_90d,search_volume
0,content_c8e9d6ab9013,client_19581e27de,page_1,9.7,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,208678,20.0
1,content_f986bd514b6e,client_7f2253d7e2,page_1,6.6,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,22456,0.0
2,content_825a9788af8d,client_4e07408562,page_1,5.6,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,16786,210.0
3,content_8ba781dafa55,client_8527a891e2,page_1,9.0,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,16156,2900.0
4,content_5d5653c4eb4f,client_4e07408562,page_1,5.7,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,15101,20.0
5,content_847a841969a2,client_19581e27de,page_1,7.4,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,14519,320.0
6,content_c82bc0c24241,client_f369cb89fc,page_1,4.3,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,13676,10.0
7,content_9983d31c53cb,client_4e07408562,page_1,5.5,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,7737,20.0
8,content_d3aaf7d5f2fc,client_19581e27de,page_1,8.3,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,7732,10.0
9,content_5195668f06db,client_19581e27de,page_1,5.2,0.0,0.24,0.24,0.24,ctr_below_tier_peers,review_title_and_meta_description,6635,0.0


**What I did:** I turned my rule into a real score. For every eligible page, score = how far its CTR falls below its own position tier's median (floored at 0, so pages already beating their tier don't get flagged). I ranked everything by score, breaking ties by impressions_90d since many pages land on the same score. 12,023 pages were eligible, and 5,888 scored above 0 — meaning about half of my visible pages are underperforming their tier.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Before I review the top 10 by hand, I want to check one thing:** does my top 10 
accidentally come from just one or two clients? My rule scores each page on its 
own, so it has no idea if it's repeating the same client's problem multiple times.

In [12]:
print(ranked.head(10)['client_id'].value_counts())

client_id
client_19581e27de    4
client_4e07408562    3
client_7f2253d7e2    1
client_8527a891e2    1
client_f369cb89fc    1
Name: count, dtype: int64


**What this tells me:** 7 rows out of my top 10 rows come from just two clients, client_19581e27de shows up 4 times and client_4e07408562 shows up 3 times. Only 3 rows represent genuinely different clients.

My rule looks at each page by itself. It doesn't know if it's picking the same client's problem over and over. It doesn't mean that my rule is wrong but instead it means that these pages really do have 0% CTR and the list is less varied than  how "top 10" should sound like it. I'll mention this again in my weak-picks section. A better version of this rule can limit how many pages one client can take up.

In [10]:
print(ranked.head(10)[['content_id', 'search_volume']])

             content_id  search_volume
0  content_c8e9d6ab9013           20.0
1  content_f986bd514b6e            0.0
2  content_825a9788af8d          210.0
3  content_8ba781dafa55         2900.0
4  content_5d5653c4eb4f           20.0
5  content_847a841969a2          320.0
6  content_c82bc0c24241           10.0
7  content_9983d31c53cb           20.0
8  content_d3aaf7d5f2fc           10.0
9  content_5195668f06db            0.0


I'm doing **top 10** review as my task on portal said the top 10 review. 
 

**Top 10 review**

**1. content_c8e9d6ab9013 (client_19581e27de)**: 

position 9.7,

208,678 impressions,

CTR 0.0, search volume 20.0

Why: By far my biggest raw-traffic gap, zero clicks on the page that gets the most visibility in my whole list.

What would make it wrong: 200K+ impressions with exactly 0 clicks is extreme. Before touching the title, I'd check this isn't a tracking or logging error. That's real page that rarely gets zero clicks at that volume.



**2. content_f986bd514b6e (client_7f2253d7e2)**:

position 6.6, 

22,456 impressions, 

CTR 0.0, search_volume 0

Why: the CTR is below tier just like every row here. 

What would make it wrong: search_volume 0 means the keyword I'm tracking has almost no measured demand, this page's real traffic is probably coming from other queries entirely, so fixing this one title might not have any impact.



**3. content_825a9788af8d (client_4e07408562)**: 

position 5.6,

16,786 impressions,

CTR 0.0, search volume 210.0

Why: strong position, zero clicks.

What would make it wrong: a good rank with zero clicks can mean a different URL on the same site is winning the click instead, it is  worth checking sibling pages before assuming the title is the problem.



**4. content_8ba781dafa55 (client_8527a891e2)**:

position 9.0, 

16,156 impressions, 

CTR 0.0, search_volume 2,900 (the highest in my top 10).

Why: real demand, real visibility, zero clicks this is the cleanest case in my list. 

What would make it wrong: honestly, this is my most confident pick. I'd send this one to a reviewer first.



**5. content_5d5653c4eb4f (client_4e07408562)**:

position 5.7,

15,101 impressions,

CTR 0.0, search volume 20.0

Why: same pattern as row 3. 

What would make it wrong: same client as row 3, this could be one client's site-wide template bug showing up twice, not two individual problems. I would want a reviewer to check row 3 first and see if fixing row 3 also fixes this one.



**6. content_847a841969a2 (client_19581e27de)**:

position 7.4, 

14,519 impressions, 

CTR 0.0, search volume 320.0

Why: standard pattern. 

What would make it wrong: this ties back to my Signal B finding (OPPOSITE), I already know high traffic 
doesn't guarantee a real opportunity here, and combined with a 90-day-only window, this could be a seasonal topic that's simply quiet right now, not permanently broken.



**7. content_c82bc0c24241 (client_f369cb89fc)**:

position 4.3,

13,676 impressions, 

CTR 0.0, search volume 10.0

Why: my best-ranked page with zero clicks. 

What would make it wrong: This page ranks really well (4.3) but still got zero clicks. That's strange for such a good position. One possible reason that can exist is that maybe Google is showing something else above it, like a quick-answer box that is already answering the question, so people don't need to click through to the page.



**8. content_9983d31c53cb (client_4e07408562)**:

position 5.5,

7,737 impressions,

CTR 0.0. search volume 20.0

Why: standard pattern. 

What would make it wrong: third row from this client (with 3 and 5), at this point I'd flag the client itself for review, not just this individual page.



**9. content_d3aaf7d5f2fc (client_19581e27de)**:

position 8.3, 

7,732 impressions, 

CTR 0.0, search volume 10.0

Why: standard pattern. 

What would make it wrong: third row from this client (with 1 and 6), and the smallest traffic of the three if a reviewer only has time for one, row 1 has ten times the impressions and should come first.



**10. content_5195668f06db (client_19581e27de)**:

position 5.2, 

6,635 impressions, 

CTR 0.0, search_volume 0.

Why: standard pattern. 

What would make it wrong: fourth row from this client, and zero search volume like row 2, the weakest pick in my top 10 on two separate counts, I'd put this one last in a reviewer's queue.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.